<div align="center">

<img src="../assets/logo.png" width="400">

# AUDITORÍA A LA CONTRATACIÓN PÚBLICA  
## Auditoria a la contratacion publica Plataforma SIA Observa – Vigencia 2025


**Entidad auditada:** 53 Sujetos de control 

**Tipo de análisis:** Auditoria basada en datos 

**Realizado por:** Wilmer Fidel Restrepo Orrego 
 
**Fecha de creacion:** 2026

**Fuente de los datos:** Plataforma Sia Observa 

</div>

Se cargan las librerías necesarias, se configuran las rutas del proyecto y se vinculan los módulos de limpieza y análisis de datos.

Se inicializa el entorno de trabajo y se muestra:

Información del sistema (SO, Python, fecha)
Versiones de las librerías
Rutas principales del proyecto

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
from datetime import datetime
import seaborn as sns

from IPython.display import display,Markdown




sys.path.append(os.path.abspath(os.path.join('..')))
from src.cleaners import aplicar_tipos_datos, TIPOS_BASICO, TIPOS_EXTENDIDO, estandarizar_modalidades,estandarizar_causales,estandarizar_recursos_v2

from src import (
    setup_environment, RAW_DATA_PATH, BASE_DIR,
    aplicar_tipos_datos, TIPOS_BASICO, TIPOS_EXTENDIDO,
    formato_moneda_co, formato_miles_co,
    calcular_alertas_adicion,calcular_duracion_vigencia
)


from src.environment import get_environment_info
from src.system import get_library_versions

setup_environment()
env = get_environment_info()
versions = get_library_versions()


print("""
AUDITORIA A LA CONTRATACION PUBLICA VIGENCIA 2025 -PLATAFORMA SIA OBSERVA
========================================================================
- Entorno configurado
- Los módulos de análisis y limpieza fueron vinculados
- Sistema listo para procesar datos de auditoria
""")

print(f"""
INFORMACION SOBRE EL ENTORNO DE DESARROLLO
=========================================================================
- Sistema operativo : {env['os']}  
- Version Python    : {env['python']}  
- Kernel            : {env['kernel']}  
- Fecha ejecucion   : {env['timestamp']}
""")

print("""
VERSION DE LAS LIBRERIAS DEL PROYECTO
==========================================================================      
""")
for lib, version in versions.items():
    print (f"{lib.capitalize():<12}:{version}")


display(Markdown (f"""
Rutas del proyecto
===========================================================================
Base del proyecto:{BASE_DIR}  
Ubicacion de las muestras de auditoria:{RAW_DATA_PATH}
"""))


AUDITORIA A LA CONTRATACION PUBLICA VIGENCIA 2025 -PLATAFORMA SIA OBSERVA
- Entorno configurado
- Los módulos de análisis y limpieza fueron vinculados
- Sistema listo para procesar datos de auditoria


INFORMACION SOBRE EL ENTORNO DE DESARROLLO
- Sistema operativo : Linux 6.17.0-20-generic  
- Version Python    : 3.12.3  
- Kernel            : CPython  
- Fecha ejecucion   : 2026-04-23 11:20:24


VERSION DE LAS LIBRERIAS DEL PROYECTO

Pandas      :3.0.0
Numpy       :2.4.2
Matplotlib  :3.10.8
Plotly      :6.5.2
Seaborn     :0.13.2



Rutas del proyecto
===========================================================================
Base del proyecto:/home/wilo/Escritorio/auditoria_sia  
Ubicacion de las muestras de auditoria:/home/wilo/Escritorio/auditoria_sia/data/raw


Proceso ETL: Limpieza y Preparación de Datos

En este bloque se ejecuta el proceso de Extracción, Transformación y Carga (ETL). El objetivo es convertir los reportes crudos de la plataforma SIA OBSERVA en un conjunto de datos estandarizado y optimizado para el análisis de auditoría.


**Ingesta y Tipado:** Carga de archivos y conversión estricta de formatos (fechas y números).
**Normalización:** Homologación de catálogos (modalidades, causales y recursos) y cálculo de métricas base.
**Aseguramiento (QA):** Verificación de integridad, control de valores nulos y monitoreo de eficiencia en memoria.

In [2]:
def procesar_pipeline_auditoria(df):

    df = estandarizar_modalidades(df)
    df = estandarizar_causales(df)
    df = estandarizar_recursos_v2(df)
    df = calcular_duracion_vigencia(df)

    return df


def realizar_qa_senior(df, nombre):

    print(f"REPORTE TÉCNICO DE INTEGRIDAD: {nombre}")
    print(f"{'='*50}")

    nulos = df.isna().sum()
    reporte_nulos = nulos[nulos > 0]

    memoria_mb = df.memory_usage(deep=True).sum() / (1024**2)

    if reporte_nulos.empty:
        print("No se detectan valores nulos.")
    else:
        print("COLUMNAS CON VALORES NULOS:")
        print(reporte_nulos.sort_values(ascending=False))

    print(f"\n MÉTRICAS DE CARGA:")
    print(f"- Total de filas: {len(df):,}")
    print(f"- Total de columnas:  {len(df.columns)}")
    print(f"- Uso de memoria:     {memoria_mb:.2f} MB")
    print(f"{'='*50}\n")


print("Cargando archivos desde RAW_DATA_PATH...")


df_basico = procesar_pipeline_auditoria(
    aplicar_tipos_datos(
        pd.read_excel(os.path.join(RAW_DATA_PATH, "Informe_Basico.xlsx")), TIPOS_BASICO
    )
)

df_ext = procesar_pipeline_auditoria(
    aplicar_tipos_datos(
        pd.read_excel(os.path.join(RAW_DATA_PATH, "Informe_Extendido.xlsx")),
        TIPOS_EXTENDIDO,
    )
)

print(f" PROCESAMIENTO COMPLETADO | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


realizar_qa_senior(df_basico, "INFORME BÁSICO")
realizar_qa_senior(df_ext, "INFORME EXTENDIDO")


print("VISTA PREVIA DE DATOS NORMALIZADOS")
display(df_basico.head(3))
display(df_ext.head(3))

Cargando archivos desde RAW_DATA_PATH...
 PROCESAMIENTO COMPLETADO | 2026-04-23 11:21:05
REPORTE TÉCNICO DE INTEGRIDAD: INFORME BÁSICO
COLUMNAS CON VALORES NULOS:
FECHA_TERMINACION_AMPLIADA    10252
NIT_1                             3
NOMBRE                            3
TIPO                              3
dtype: int64

 MÉTRICAS DE CARGA:
- Total de filas: 14,199
- Total de columnas:  25
- Uso de memoria:     13.21 MB

REPORTE TÉCNICO DE INTEGRIDAD: INFORME EXTENDIDO
No se detectan valores nulos.

 MÉTRICAS DE CARGA:
- Total de filas: 14,138
- Total de columnas:  33
- Uso de memoria:     23.44 MB

VISTA PREVIA DE DATOS NORMALIZADOS


,TIPO_DE_ENTIDAD,NIT,ENTIDAD,VIGENCIA,CODIGO_CONTRATO,VALOR_INICIAL_CONTRATO,ADICIONES,LIBERACIONES,VALOR_VIGENTE,FECHA_SUSCRIPCION,FECHA_ACTA_DE_INICIO,FECHA_TERMINACION,TIEMPO_EJECUCION,MODALIDAD_CONTRATACION,CAUSAL_CONTRATO,TIPO_CONTRATO,FECHA_CREACION,FECHA_TERMINACION_AMPLIADA,NIT_1,NOMBRE,TIPO,ESTADO_CONTRATO,MODALIDAD_ESTANDAR,CAUSAL_ESTANDAR,DURACION_REAL
0,SOCIEDADES DE ECONOMIA MIXTA,901685734,UMBRALED S.A.S E.S.P.,2025,UMB-01-2025,"17,676,000.00",0.00,0.00,"17,676,000.00",2025-01-01,2025-01-01,2025-12-31,364,CONTRATACION DIRECTA,PRESTACION DE SERVICIOS PROFESIONALES Y APOYO,APOYO A LA GESTION,2025-01-31,NaT,24547202,MARIA LILIANA MONTES HOYOS,Contratista,RENDIDO,CONTRATACION DIRECTA,PRESTACION DE SERVICIOS PROFESIONALES Y APOYO,364
1,SOCIEDADES DE ECONOMIA MIXTA,901685734,UMBRALED S.A.S E.S.P.,2025,UMB-02-2025,"8,846,800.00",0.00,0.00,"8,846,800.00",2025-01-01,2025-01-01,2025-12-31,364,CONTRATACION DIRECTA,PRESTACION DE SERVICIOS PROFESIONALES Y APOYO,APOYO A LA GESTION,2025-01-31,NaT,900010878,GRUPO METRO Y CIA. LTDA,Contratista,RENDIDO,CONTRATACION DIRECTA,PRESTACION DE SERVICIOS PROFESIONALES Y APOYO,364
2,SOCIEDADES DE ECONOMIA MIXTA,901685734,UMBRALED S.A.S E.S.P.,2025,UMB-03-2025,"22,176,000.00",0.00,0.00,"22,176,000.00",2025-01-02,2025-01-02,2025-12-31,363,CONTRATACION DIRECTA,PRESTACION DE SERVICIOS PROFESIONALES Y APOYO,APOYO A LA GESTION,2025-01-31,NaT,1088290219,DIANA MARCELA RIOS AGUIRRE,Contratista,RENDIDO,CONTRATACION DIRECTA,PRESTACION DE SERVICIOS PROFESIONALES Y APOYO,363


,TIPO_DE_ENTIDAD,NIT,ENTIDAD,VIGENCIA,CODIGO_CONTRATO,OBJETO_CONTRATO,FECHA_SUSCRIPCION,FECHA_ACTA_DE_INICIO,FECHA_TERMINACION,TIEMPO_EJECUCION,VALOR_INICIAL_CONTRATO,ADICIONES,LIBERACIONES,VALOR_VIGENTE,MODALIDAD_CONTRATACION,CAUSAL_CONTRATO,TIPO_CONTRATO,NIT_1,NOMBRE,TIPO,NIT_2,NOMBRE_1,TIPO_1,NOMBRE_DEL_RUBRO,APROPIACION_INICIAL,ORIGEN_RECURSOS,CDPS,RPS,FECHA_CREACION,MODALIDAD_ESTANDAR,CAUSAL_ESTANDAR,ORIGEN_RECURSOS_ESTANDAR,DURACION_REAL
0,SOCIEDADES DE ECONOMIA MIXTA,901685734,UMBRALED S.A.S E.S.P.,2025,UMB-01-2025,ARRENDAMIENTO DE INMUEBLE CON DESTINACION COME...,2025-01-01,2025-01-01,2025-12-31,364,"17,676,000.00",0.00,0.00,"17,676,000.00",CONTRATACION DIRECTA,PRESTACION DE SERVICIOS PROFESIONALES Y APOYO,APOYO A LA GESTION,24547202,MARIA LILIANA MONTES HOYOS,Contratista,1088332330,JORGE WILLIAM GONZALEZ TAMAYO,Interno,SERVICIOS FINANCIEROS Y SERVICIOS CONEXOS; SER...,"68,266,800.00",RECURSOS PROPIOS,20250001,20250001,2025-01-31,CONTRATACION DIRECTA,PRESTACION DE SERVICIOS PROFESIONALES Y APOYO,RECURSOS PROPIOS,364
1,SOCIEDADES DE ECONOMIA MIXTA,901685734,UMBRALED S.A.S E.S.P.,2025,UMB-02-2025,"CONTRATAR LA IMPLEMENTACION, PUESTA EN FUNCION...",2025-01-01,2025-01-01,2025-12-31,364,"8,846,800.00",0.00,0.00,"8,846,800.00",CONTRATACION DIRECTA,PRESTACION DE SERVICIOS PROFESIONALES Y APOYO,APOYO A LA GESTION,900010878,GRUPO METRO Y CIA. LTDA,Contratista,1088332330,JORGE WILLIAM GONZALEZ TAMAYO,Interno,PAQUETES DE SOFTWARE,"9,072,000.00",RECURSOS PROPIOS,20250003,20250003,2025-01-31,CONTRATACION DIRECTA,PRESTACION DE SERVICIOS PROFESIONALES Y APOYO,RECURSOS PROPIOS,364
2,SOCIEDADES DE ECONOMIA MIXTA,901685734,UMBRALED S.A.S E.S.P.,2025,UMB-03-2025,PRESTAR EL SERVICIO PROFESIONAL EN EL CARGO DE...,2025-01-02,2025-01-02,2025-12-31,363,"22,176,000.00",0.00,0.00,"22,176,000.00",CONTRATACION DIRECTA,PRESTACION DE SERVICIOS PROFESIONALES Y APOYO,APOYO A LA GESTION,1088290219,DIANA MARCELA RIOS AGUIRRE,Contratista,1088332330,JORGE WILLIAM GONZALEZ TAMAYO,Interno,SERVICIOS PRESTADOS A LAS EMPRESAS Y SERVICIOS...,"171,903,986.00",RECURSOS PROPIOS,20250005,20250003,2025-01-31,CONTRATACION DIRECTA,PRESTACION DE SERVICIOS PROFESIONALES Y APOYO,RECURSOS PROPIOS,363


Generación de Entregables y Origen para Power BI

En este bloque se consolidan los resultados del proceso ETL en la carpeta de datos procesados (`/data/processed`). Estos archivos constituyen la fuente de verdad que será consumida por el tablero de control en Power BI.

Optimizaciones para Analítica y Visualización:

Compatibilidad de Power Query: Se generan archivos CSV con codificación `utf-8-sig` y separador `;`, optimizados para una carga limpia en Power BI sin errores de delimitación o caracteres.

Normalización de Fechas: Las columnas cronológicas se exportan en formato ISO (`YYYY-MM-DD`), facilitando la creación de tablas de calendario y jerarquías temporales en DAX.

Integridad de Datos: El uso de copias temporales (`.copy()`) garantiza que el formato de exportación para consumo externo no altere los tipos de datos lógicos dentro del entorno de Python.

In [3]:
# BLOQUE DE EXPORTACIÓN DE DATOS PROCESADOS (EXCEL PARA HERRAMIENTA DE VISUALIZACION)
from src.system import exportar_para_bi
import os

print("Exportando archivos limpios para Power BI")

PROCESSED_DATA_PATH = os.path.join("..", "data", "processed")


archivos_a_exportar = {
    "Informe_Basico_Procesado": df_basico,
    "Informe_Extendido_Procesado": df_ext,
}

try:
    exportar_para_bi(archivos_a_exportar, PROCESSED_DATA_PATH)
    print(f"\n Ruta: {os.path.abspath(PROCESSED_DATA_PATH)}")
except Exception as e:
    print(f"Error en exportación: {e}")

Exportando archivos limpios para Power BI
Exportado: Informe_Basico_Procesado.csv
Exportado: Informe_Extendido_Procesado.csv

 Ruta: /home/wilo/Escritorio/auditoria_sia/data/processed


In [4]:
from src.analysis import calcular_resumen_contratos  # Asumiendo que esa es tu ruta

# Invocas la función
resumen = calcular_resumen_contratos(df_ext)

print(f"📊 --- RESULTADOS DE AUDITORÍA ---")
print(f"✅ Total Contratos: {resumen['cantidad']}")
print(f"💰 Valor Total: ${resumen['total_suma']:,.2f}")
print(f"📉 Valor Promedio: ${resumen['promedio']:,.2f}")
print("-" * 40)
print(f"🏆 CONTRATO TOP (Mayor Valor):")
print(f"🏢 Entidad: {resumen['mayor_contrato']['entidad']}")
print(f"🆔 Código: {resumen['mayor_contrato']['codigo']}")
print(f"💵 Valor: ${resumen['mayor_contrato']['valor']:,.2f}")
print(f"💵 objeto: {resumen['mayor_contrato']['objeto']}")

📊 --- RESULTADOS DE AUDITORÍA ---
✅ Total Contratos: 14138
💰 Valor Total: $1,329,649,976,891.29
📉 Valor Promedio: $94,047,954.23
----------------------------------------
🏆 CONTRATO TOP (Mayor Valor):
🏢 Entidad: EMPRESA TERRITORIAL DE DESARROLLO URBANO Y RURAL DE RISARALDA
🆔 Código: 307-2025
💵 Valor: $98,405,360,859.00
💵 objeto: CONSTRUCCION DE LA ESTRUCTURA DEL EDIFICO DE ALTA COMPLEJIDAD DEL PROYECTO HOSPITAL REGIONAL DE ALTA COMPLEJIDAD EN EL DEPARTAMENTO DE RISARALDA


In [5]:
from src.analysis import contratacion_por_tipo_entidad

# Ejecutamos el análisis por grupo
df_por_entidad = contratacion_por_tipo_entidad(df_ext)

print("🏢 ANÁLISIS POR TIPO DE ENTIDAD")
display(df_por_entidad)

# Si quieres ver solo la entidad con más plata:
entidad_top = df_por_entidad.iloc[0]["TIPO_DE_ENTIDAD"]
print(f"\n📢 La mayor inversión está en: {entidad_top}")

🏢 ANÁLISIS POR TIPO DE ENTIDAD


,TIPO_DE_ENTIDAD,CANTIDAD_CONTRATOS,VALOR_TOTAL
5,ENTIDADES TERRITORIALES,7609,"661,975,700,550.35"
4,EMPRESAS SOCIALES DEL ESTADO,4464,"382,860,344,191.53"
3,EMPRESAS INDUSTRIALES Y COMERCIALES DEL ESTADO,569,"181,785,600,313.00"
2,EMPRESA DE SERVICIOS PUBLICOS,942,"84,637,736,788.55"
0,AREAS METROPOLITANAS,361,"14,254,334,544.86"
6,ESTABLECIMIENTO PUBLICOS,123,"2,739,627,498.00"
1,ASOCIACION ENTRE ENTIDADES PUBLICAS,49,"984,830,573.00"
7,SOCIEDADES DE ECONOMIA MIXTA,21,"411,802,432.00"



📢 La mayor inversión está en: ENTIDADES TERRITORIALES


In [ ]:
# 1. Obtener los datos agrupados
df_grafica = contratacion_por_tipo_entidad(df_ext)

# 2. Configuración de la figura
plt.figure(figsize=(14, 8))
ax = sns.barplot(
    data=df_grafica,
    x="VALOR_TOTAL",
    y="TIPO_DE_ENTIDAD",
    palette="viridis",
    hue="TIPO_DE_ENTIDAD",
    legend=False,
)

# 3. EL TRUCO: Agregar etiquetas personalizadas a cada barra
for i, bar in enumerate(ax.patches):
    # Obtenemos los valores de esa fila
    monto = df_grafica.iloc[i]["VALOR_TOTAL"]
    cantidad = df_grafica.iloc[i]["CANTIDAD_CONTRATOS"]

    # Creamos el texto formateado
    # Ejemplo: 222 contratos | $2.222.222.222
    label_text = f" {int(cantidad):,} contratos | ${monto:,.0f}".replace(",", ".")

    # Posicionamos el texto al final de la barra
    ax.text(
        bar.get_width(),  # Posición X (final de la barra)
        bar.get_y() + bar.get_height() / 2,  # Posición Y (centro de la barra)
        label_text,
        va="center",  # Alineación vertical
        fontsize=10,
        fontweight="bold",
        color="#333333",
    )

# 4. Limpieza estética
plt.title(
    "💰 Inversión Total y Volumen por Tipo de Entidad",
    fontsize=16,
    fontweight="bold",
    pad=20,
)
plt.xlabel("")  # Quitamos el título del eje X
plt.ylabel("")  # Quitamos el título del eje Y para que no estorbe
plt.xticks([])  # QUITAMOS LOS NÚMEROS AMONTONADOS DEL EJE X

# 5. Ajustar márgenes para que el texto largo quepa
plt.xlim(
    0, df_grafica["VALOR_TOTAL"].max() * 1.5
)  # Damos un 50% más de espacio a la derecha
sns.despine(
    left=True, bottom=True
)  # Quitamos los bordes de la gráfica para que se vea moderna

plt.tight_layout()
plt.show()

In [ ]:
from src.analysis import analizar_por_tipo_contrato

# Obtener datos
df_tipo_contrato = analizar_por_tipo_contrato(df_ext)

print("📋 TABLA RESUMEN: PARTICIPACIÓN POR TIPO DE CONTRATO")
# Formateamos la tabla para que se vea bonita al mostrarla
df_estilo = df_tipo_contrato.copy()
df_estilo["VALOR_TOTAL"] = (
    df_estilo["VALOR_TOTAL"].map("${:,.0f}".format).str.replace(",", ".")
)
df_estilo["PORCENTAJE_PARTICIPACION"] = df_estilo["PORCENTAJE_PARTICIPACION"].map(
    "{:.2f}%".format
)
display(df_estilo)

In [ ]:
plt.figure(figsize=(12, 6))
ax = sns.barplot(
    data=df_tipo_contrato, x="VALOR_TOTAL", y="TIPO_CONTRATO", palette="magma"
)

# Anotamos el dinero en cada barra
for i, bar in enumerate(ax.patches):
    monto = df_tipo_contrato.iloc[i]["VALOR_TOTAL"]
    ax.text(
        bar.get_width(),
        bar.get_y() + bar.get_height() / 2,
        f"  ${monto:,.0f}".replace(",", "."),
        va="center",
        fontweight="bold",
    )

plt.title("💰 Inversión Total por Tipo de Contrato", fontsize=15, fontweight="bold")
plt.xlabel("Valor Total")
plt.ylabel("")
plt.xticks([])  # Quitamos los números del eje X
sns.despine(left=True, bottom=True)
plt.show()

In [ ]:
# 1. Obtenemos los datos agrupados usando la función que ya definimos
df_tipo_contrato = analizar_por_tipo_contrato(df_ext)

# 2. Configuración de la figura para que sea legible
plt.figure(figsize=(14, 10))  # Más alta para dar espacio a todas las categorías

# Usamos barras horizontales para que los nombres largos se lean de una
ax = sns.barplot(
    data=df_tipo_contrato,
    x="PORCENTAJE_PARTICIPACION",  # Porcentaje en el eje X
    y="TIPO_CONTRATO",  # Categorías en el eje Y
    palette="viridis",  # Degradado profesional
    hue="TIPO_CONTRATO",
    legend=False,
)

# 3. EL TRUCO DE AUDITOR: Anotar el porcentaje exacto al final de cada barra
for bar in ax.patches:
    # Obtenemos el valor porcentual de esa barra
    porcentaje = bar.get_width()

    # Creamos la etiqueta con el formato X.X%
    label_text = f" {porcentaje:.1f}%".replace(".", ",")  # Usamos coma para decimales

    # Posicionamos el texto
    ax.text(
        bar.get_width(),  # Posición X (final de la barra)
        bar.get_y() + bar.get_height() / 2,  # Posición Y (centro)
        label_text,
        va="center",  # Alineación vertical
        fontsize=10,
        fontweight="bold",
        color="#333333",
    )

# 4. Ajustes estéticos y de Limpieza
plt.title(
    "📊 Distribución Porcentual del Volumen de Contratos",
    fontsize=16,
    fontweight="bold",
    pad=20,
)
plt.xlabel("")  # Quitamos el título del eje X
plt.ylabel("")  # Quitamos el título del eje Y
plt.xticks([])  # QUITAMOS LOS NÚMEROS AMONTONADOS DEL EJE X

# 5. Dar aire y limpiar bordes
plt.xlim(
    0, df_tipo_contrato["PORCENTAJE_PARTICIPACION"].max() * 1.15
)  # 15% más de espacio a la derecha
sns.despine(left=True, bottom=True)  # Quitamos los bordes de la gráfica

plt.tight_layout()
plt.show()

In [ ]:
from src.analysis import analizar_por_modalidad

# 1. Obtener los datos
df_modalidad = analizar_por_modalidad(df_ext)

# 2. Configuración de la gráfica
plt.figure(figsize=(14, 8))
ax = sns.barplot(
    data=df_modalidad,
    x="VALOR_TOTAL",
    y="MODALIDAD_ESTANDAR",
    palette="magma",
    hue="MODALIDAD_ESTANDAR",
    legend=False,
)

# 3. Anotaciones de Auditoría (Cantidad | Valor | %)
for i, bar in enumerate(ax.patches):
    monto = df_modalidad.iloc[i]["VALOR_TOTAL"]
    cant = df_modalidad.iloc[i]["CANTIDAD"]
    porc = df_modalidad.iloc[i]["%_PARTICIPACION"]

    # Texto formateado para Colombia (Puntos para miles)
    label = f" {int(cant):,} contr. | ${monto:,.0f} | {porc:.1f}%".replace(",", ".")

    ax.text(
        bar.get_width(),
        bar.get_y() + bar.get_height() / 2,
        label,
        va="center",
        fontweight="bold",
        fontsize=10,
    )

# 4. Estética final de la gráfica
plt.title(
    "🏛️ Análisis de Contratación por Modalidad (Estandarizada)",
    fontsize=16,
    fontweight="bold",
    pad=20,
)
plt.xlabel("")
plt.ylabel("")
plt.xticks([])
plt.xlim(0, df_modalidad["VALOR_TOTAL"].max() * 1.6)
sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.show()

# 5. TABLA RESUMEN CON ESTILO (Mejorada)
print("\n" + "=" * 50)
print("📋 TABLA RESUMEN DE MODALIDADES")
print("=" * 50)

# Aplicamos estilos: Fondo de color para resaltar montos y barras de progreso visuales
tabla_estilizada = (
    df_modalidad.style.format(
        {
            "VALOR_TOTAL": lambda x: f"$ {x:,.0f}".replace(",", "."),
            "%_PARTICIPACION": "{:.2f}%",
        }
    )
    .bar(subset=["VALOR_TOTAL"], color="")
    .set_properties(**{"text-align": "left", "font-family": "Arial"})
    .set_table_styles(
        [
            {
                "selector": "th",
                "props": [("background-color", "#2c3e50"), ("color", "white")],
            }
        ]
    )
)

display(tabla_estilizada)

In [ ]:
from src.analysis import analizar_por_origen_recursos

# 1. Obtener los datos calculados
df_recursos = analizar_por_origen_recursos(df_ext)

# 2. Configuración de la gráfica
plt.figure(figsize=(14, 7))
ax = sns.barplot(
    data=df_recursos,
    x="VALOR_TOTAL",
    y="ORIGEN_RECURSOS_ESTANDAR",
    palette="viridis",
    hue="ORIGEN_RECURSOS_ESTANDAR",
    legend=False,
)

# 3. Anotaciones detalladas (Cantidad | Valor | % Inversión)
for i, bar in enumerate(ax.patches):
    monto = df_recursos.iloc[i]["VALOR_TOTAL"]
    cant = df_recursos.iloc[i]["CANTIDAD_CONTRATOS"]
    porc = df_recursos.iloc[i]["%_INVERSION"]

    # Formato: 120 contr. | $5.000.000 | 25.4%
    label = f" {int(cant):,} contr. | ${monto:,.0f} | {porc:.1f}%".replace(",", ".")

    ax.text(
        bar.get_width(),
        bar.get_y() + bar.get_height() / 2,
        label,
        va="center",
        fontweight="bold",
        fontsize=10,
    )

# 4. Estética de la gráfica
plt.title(
    "💰 Distribución de Inversión por Origen de Recursos",
    fontsize=16,
    fontweight="bold",
    pad=25,
)
plt.xlabel("")
plt.ylabel("")
plt.xticks([])  # Limpiamos el eje X
plt.xlim(0, df_recursos["VALOR_TOTAL"].max() * 1.6)  # Espacio para el texto
sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.show()

# 5. TABLA RESUMEN ESTILIZADA
print("\n📋 RESUMEN FINANCIERO POR FUENTE DE FINANCIACIÓN:")
tabla_fuentes = (
    df_recursos.style.format(
        {
            "VALOR_TOTAL": lambda x: f"$ {x:,.0f}".replace(",", "."),
            "%_INVERSION": "{:.2f}%",
        }
    )
    .background_gradient(subset=["%_INVERSION"], cmap="Greens")
    .set_properties(**{"text-align": "left"})
)

display(tabla_fuentes)

In [ ]:
from src.analysis import analizar_ranking_entidades

# 1. Obtener los datos del ranking
df_ranking = analizar_ranking_entidades(df_ext)

# 2. Presentación de la Tabla Resumen Estilizada
print("🏆 RANKING DE ENTIDADES POR MONTO CONTRATADO")
print("-" * 50)

# Aplicamos un estilo profesional
ranking_estilizado = (
    df_ranking.head(20)
    .style.format(
        {
            "VALOR_TOTAL": lambda x: f"$ {x:,.0f}".replace(",", "."),
            "CANTIDAD_CONTRATOS": "{:,}",
        }
    )
    .background_gradient(subset=["VALOR_TOTAL"], cmap="YlOrRd")
    .set_properties(**{"text-align": "left"})
    .set_table_styles(
        [
            {
                "selector": "th",
                "props": [
                    ("background-color", "#1a5276"),
                    ("color", "white"),
                    ("font-weight", "bold"),
                ],
            }
        ]
    )
)

# Mostramos el Top 20 (puedes quitar el .head(20) si quieres ver todas)
display(ranking_estilizado)

In [ ]:
from src.analysis import calcular_alertas_adicion

# 1. Ejecutar el cálculo sobre el informe extendido
df_con_alertas = calcular_alertas_adicion(df_ext)

# 2. Capturar los que superan el 50% (Zona Roja)
# Importante: El texto debe coincidir con el de la función ("ALERTA_ROJA")
sospechosos = df_con_alertas[df_con_alertas["ESTADO_ADICION"] == "ALERTA_ROJA"].copy()

# 3. Reporte de Auditoría
print(
    f"🚨 ALERTA DE AUDITORÍA: Se detectaron {len(sospechosos)} contratos con adiciones críticas."
)

if not sospechosos.empty:
    print("\n📋 TABLA RESUMEN: CONTRATOS QUE SUPERAN EL 50% DE ADICIÓN")

    # Seleccionamos solo las columnas que pidió, senior
    reporte_final = sospechosos[
        ["ENTIDAD", "CODIGO_CONTRATO", "PORCENTAJE_ADICION"]
    ].sort_values(by="PORCENTAJE_ADICION", ascending=False)

    # Le damos formato profesional
    display(
        reporte_final.style.format(
            {"PORCENTAJE_ADICION": "{:.2f}%"}
        ).background_gradient(subset=["PORCENTAJE_ADICION"], cmap="Reds")
    )
else:
    # Si sigue saliendo 0, miremos los que tienen cualquier adición para ver si el dato existe
    print("✅ No hay contratos > 50%.")
    con_algo = df_con_alertas[df_con_alertas["PORCENTAJE_ADICION"] > 0]
    print(
        f"ℹ️ Nota: Se encontraron {len(con_algo)} contratos con adiciones menores al 50%."
    )

In [ ]:
from src.analysis import (
    analizar_rendicion_extemporanea,
    resumen_extemporaneos_por_entidad,
)

# 1. Ejecutar el cálculo sobre el INFORME BÁSICO
df_resultado = analizar_rendicion_extemporanea(df_basico)

print(f"📊 TOTAL DE CONTRATOS RENDIDOS EXTEMPORÁNEOS: {len(df_resultado)}")

# --- TABLA 1: DETALLE DE CONTRATOS ---
print("\n📋 DETALLE DE EXTEMPORANEIDAD (TOP 10 MÁS CRÍTICOS):")
columnas_detalle = [
    "ENTIDAD",
    "CODIGO_CONTRATO",
    "FECHA_ACTA_DE_INICIO",
    "FECHA_CREACION",
    "DIAS_EXTEMPORANEIDAD",
]

display(
    df_resultado[columnas_detalle]
    .sort_values(by="DIAS_EXTEMPORANEIDAD", ascending=False)
    .head(10)
    .style.background_gradient(subset=["DIAS_EXTEMPORANEIDAD"], cmap="YlOrRd")
)

# --- TABLA 2: RESUMEN POR ENTIDAD ---
print("\n🏢 RESUMEN DE INCUMPLIMIENTO POR ENTIDAD:")
df_resumen_entidad = resumen_extemporaneos_por_entidad(df_resultado)

display(
    df_resumen_entidad.style.format({"PROMEDIO_DIAS_RETRASO": "{:.1f} días"}).bar(
        subset=["TOTAL_CONTRATOS_EXTEMPORANEOS"], color="#d9534f"
    )
)

In [ ]:
# Definir el nombre del archivo
nombre_archivo = "Reporte_Extemporaneidad_SIA.xlsx"

# Creamos el escritor de Excel
with pd.ExcelWriter(nombre_archivo, engine="xlsxwriter") as writer:
    # 1. Exportamos el resumen por entidad a la primera pestaña
    df_resumen_entidad.to_excel(writer, sheet_name="Resumen_Entidades", index=False)

    # 2. Exportamos el detalle completo (todos, no solo el top 10) a la segunda pestaña
    columnas_exportar = [
        "ENTIDAD",
        "CODIGO_CONTRATO",
        "FECHA_ACTA_DE_INICIO",
        "FECHA_CREACION",
        "DIAS_EXTEMPORANEIDAD",
    ]
    df_resultado[columnas_exportar].sort_values(
        by="DIAS_EXTEMPORANEIDAD", ascending=False
    ).to_excel(writer, sheet_name="Detalle_Contratos", index=False)

    # 3. Formato Senior (Opcional pero recomendado)
    # Esto le pone un poquito de estilo a las columnas automáticamente
    workbook = writer.book
    for sheet in writer.sheets.values():
        sheet.set_column("A:Z", 20)  # Ajusta el ancho de las columnas

print(f"✅ ¡Listo, parce! El reporte ha sido exportado como: {nombre_archivo}")